# 1. Carga de datos

Notebook adaptado para Databricks Free Edition. Carga un dataset público y persiste los datos en Delta para reutilizarlos en las siguientes etapas.

In [ ]:
from pathlib import Path

import requests
from pyspark.sql import functions as F

PUBLIC_DATASET_URL = "https://raw.githubusercontent.com/plotly/datasets/master/diabetes.csv"
LOCAL_DATASET_DIR = Path("data")
LOCAL_DATASET_FILE = LOCAL_DATASET_DIR / "diabetes.csv"

LOCAL_DATASET_DIR.mkdir(parents=True, exist_ok=True)
response = requests.get(PUBLIC_DATASET_URL, timeout=30)
response.raise_for_status()
LOCAL_DATASET_FILE.write_text(response.text, encoding="utf-8")

raw_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(str(LOCAL_DATASET_FILE))
)

raw_df.printSchema()
raw_df.show(5, truncate=False)

In [ ]:
raw_df = raw_df.withColumn("ingested_at", F.current_timestamp())
raw_df.printSchema()
raw_df.show(5, truncate=False)

raw_df.write.mode("overwrite").format("delta").saveAsTable("bronze_control_2")
print("Datos cargados y persistidos en Delta como bronze_control_2")